In [0]:
!rm -rf *
!git clone https://github.com/manuhg/darknet dkn
!mv -v dkn/* ./
!mv -v dkn/.git* ./

Cloning into 'dkn'...
remote: Enumerating objects: 5940, done.
remote: Total 5940 (delta 0), reused 0 (delta 0), pack-reused 5940
Receiving objects: 100% (5940/5940), 7.98 MiB | 31.31 MiB/s, done.
Resolving deltas: 100% (3961/3961), done.
renamed 'dkn/cfg' -> './cfg'
renamed 'dkn/darknet_det.py' -> './darknet_det.py'
renamed 'dkn/data' -> './data'
renamed 'dkn/examples' -> './examples'
renamed 'dkn/include' -> './include'
renamed 'dkn/libdarknet.a' -> './libdarknet.a'
renamed 'dkn/libdarknet.so' -> './libdarknet.so'
renamed 'dkn/LICENSE' -> './LICENSE'
renamed 'dkn/LICENSE.fuck' -> './LICENSE.fuck'
renamed 'dkn/LICENSE.gen' -> './LICENSE.gen'
renamed 'dkn/LICENSE.gpl' -> './LICENSE.gpl'
renamed 'dkn/LICENSE.meta' -> './LICENSE.meta'
renamed 'dkn/LICENSE.mit' -> './LICENSE.mit'
renamed 'dkn/LICENSE.v1' -> './LICENSE.v1'
renamed 'dkn/Makefile' -> './Makefile'
renamed 'dkn/pallete' -> './pallete'
renamed 'dkn/README.md' -> './README.md'
renamed 'dkn/scripts' -> './scripts'
renamed 'dkn/sr

In [0]:
!sed -i 's/GPU=0/GPU=1/;s/CUDNN=0/CUDNN=1/;s/OPENCV=0/OPENCV=1/;s/OPENMP=0/OPENMP=1/;' Makefile
!head Makefile

GPU=1
CUDNN=1
OPENCV=1
OPENMP=1
DEBUG=0

ARCH= -gencode arch=compute_30,code=sm_30 \
      -gencode arch=compute_35,code=sm_35 \
      -gencode arch=compute_50,code=[sm_50,compute_50] \
      -gencode arch=compute_52,code=[sm_52,compute_52]


In [0]:
!make -j8

mkdir -p obj
mkdir -p backup
mkdir -p results
gcc -Iinclude/ -Isrc/ -DOPENCV `pkg-config --cflags opencv`  -DGPU -I/usr/local/cuda/include/ -DCUDNN  -Wall -Wno-unused-result -Wno-unknown-pragmas -Wfatal-errors -fPIC -fopenmp -Ofast -DOPENCV -DGPU -DCUDNN -c ./src/gemm.c -o obj/gemm.o
gcc -Iinclude/ -Isrc/ -DOPENCV `pkg-config --cflags opencv`  -DGPU -I/usr/local/cuda/include/ -DCUDNN  -Wall -Wno-unused-result -Wno-unknown-pragmas -Wfatal-errors -fPIC -fopenmp -Ofast -DOPENCV -DGPU -DCUDNN -c ./src/utils.c -o obj/utils.o
gcc -Iinclude/ -Isrc/ -DOPENCV `pkg-config --cflags opencv`  -DGPU -I/usr/local/cuda/include/ -DCUDNN  -Wall -Wno-unused-result -Wno-unknown-pragmas -Wfatal-errors -fPIC -fopenmp -Ofast -DOPENCV -DGPU -DCUDNN -c ./src/cuda.c -o obj/cuda.o
gcc -Iinclude/ -Isrc/ -DOPENCV `pkg-config --cflags opencv`  -DGPU -I/usr/local/cuda/include/ -DCUDNN  -Wall -Wno-unused-result -Wno-unknown-pragmas -Wfatal-errors -fPIC -fopenmp -Ofast -DOPENCV -DGPU -DCUDNN -c ./src/deconvolutional_l

In [0]:
!wget -nc https://pjreddie.com/media/files/yolov3-tiny.weights

--2019-02-15 10:43:26--  https://pjreddie.com/media/files/yolov3-tiny.weights
Resolving pjreddie.com (pjreddie.com)... 128.208.3.39
Connecting to pjreddie.com (pjreddie.com)|128.208.3.39|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35434956 (34M) [application/octet-stream]
Saving to: ‘yolov3-tiny.weights’

yolov3-tiny.weights 100%[===================>]  33.79M  91.2MB/s    in 0.4s    

2019-02-15 10:43:27 (91.2 MB/s) - ‘yolov3-tiny.weights’ saved [35434956/35434956]



In [0]:
!./darknet detect cfg/yolov3-tiny.cfg yolov3-tiny.weights data/dog.jpg

layer     filters    size              input                output
    0 conv     16  3 x 3 / 1   416 x 416 x   3   ->   416 x 416 x  16  0.150 BFLOPs
    1 max          2 x 2 / 2   416 x 416 x  16   ->   208 x 208 x  16
    2 conv     32  3 x 3 / 1   208 x 208 x  16   ->   208 x 208 x  32  0.399 BFLOPs
    3 max          2 x 2 / 2   208 x 208 x  32   ->   104 x 104 x  32
    4 conv     64  3 x 3 / 1   104 x 104 x  32   ->   104 x 104 x  64  0.399 BFLOPs
    5 max          2 x 2 / 2   104 x 104 x  64   ->    52 x  52 x  64
    6 conv    128  3 x 3 / 1    52 x  52 x  64   ->    52 x  52 x 128  0.399 BFLOPs
    7 max          2 x 2 / 2    52 x  52 x 128   ->    26 x  26 x 128
    8 conv    256  3 x 3 / 1    26 x  26 x 128   ->    26 x  26 x 256  0.399 BFLOPs
    9 max          2 x 2 / 2    26 x  26 x 256   ->    13 x  13 x 256
   10 conv    512  3 x 3 / 1    13 x  13 x 256   ->    13 x  13 x 512  0.399 BFLOPs
   11 max          2 x 2 / 1    13 x  13 x 512   ->    13 x  13 x 512
   12 con

In [0]:
from ctypes import *
import math
import random
import cv2
def sample(probs):
    s = sum(probs)
    probs = [a/s for a in probs]
    r = random.uniform(0, 1)
    for i in range(len(probs)):
        r = r - probs[i]
        if r <= 0:
            return i
    return len(probs)-1

def c_array(ctype, values):
    arr = (ctype*len(values))()
    arr[:] = values
    return arr

class BOX(Structure):
    _fields_ = [("x", c_float),
                ("y", c_float),
                ("w", c_float),
                ("h", c_float)]

class DETECTION(Structure):
    _fields_ = [("bbox", BOX),
                ("classes", c_int),
                ("prob", POINTER(c_float)),
                ("mask", POINTER(c_float)),
                ("objectness", c_float),
                ("sort_class", c_int)]


class IMAGE(Structure):
    _fields_ = [("w", c_int),
                ("h", c_int),
                ("c", c_int),
                ("data", POINTER(c_float))]

class METADATA(Structure):
    _fields_ = [("classes", c_int),
                ("names", POINTER(c_char_p))]

    

lib = CDLL("./libdarknet.so", RTLD_GLOBAL)
lib.network_width.argtypes = [c_void_p]
lib.network_width.restype = c_int
lib.network_height.argtypes = [c_void_p]
lib.network_height.restype = c_int

predict = lib.network_predict
predict.argtypes = [c_void_p, POINTER(c_float)]
predict.restype = POINTER(c_float)

set_gpu = lib.cuda_set_device
set_gpu.argtypes = [c_int]

make_image = lib.make_image
make_image.argtypes = [c_int, c_int, c_int]
make_image.restype = IMAGE

get_network_boxes = lib.get_network_boxes
get_network_boxes.argtypes = [c_void_p, c_int, c_int, c_float, c_float, POINTER(c_int), c_int, POINTER(c_int)]
get_network_boxes.restype = POINTER(DETECTION)

make_network_boxes = lib.make_network_boxes
make_network_boxes.argtypes = [c_void_p]
make_network_boxes.restype = POINTER(DETECTION)

free_detections = lib.free_detections
free_detections.argtypes = [POINTER(DETECTION), c_int]

free_ptrs = lib.free_ptrs
free_ptrs.argtypes = [POINTER(c_void_p), c_int]

network_predict = lib.network_predict
network_predict.argtypes = [c_void_p, POINTER(c_float)]

reset_rnn = lib.reset_rnn
reset_rnn.argtypes = [c_void_p]

load_net = lib.load_network
load_net.argtypes = [c_char_p, c_char_p, c_int]
load_net.restype = c_void_p

do_nms_obj = lib.do_nms_obj
do_nms_obj.argtypes = [POINTER(DETECTION), c_int, c_int, c_float]

do_nms_sort = lib.do_nms_sort
do_nms_sort.argtypes = [POINTER(DETECTION), c_int, c_int, c_float]

free_image = lib.free_image
free_image.argtypes = [IMAGE]

letterbox_image = lib.letterbox_image
letterbox_image.argtypes = [IMAGE, c_int, c_int]
letterbox_image.restype = IMAGE

load_meta = lib.get_metadata
lib.get_metadata.argtypes = [c_char_p]
lib.get_metadata.restype = METADATA

load_image = lib.load_image_color
load_image.argtypes = [c_char_p, c_int, c_int]
load_image.restype = IMAGE

rgbgr_image = lib.rgbgr_image
rgbgr_image.argtypes = [IMAGE]

predict_image = lib.network_predict_image
predict_image.argtypes = [c_void_p, IMAGE]
predict_image.restype = POINTER(c_float)

def classify(net, meta, im):
    out = predict_image(net, im)
    res = []
    for i in range(meta.classes):
        res.append((meta.names[i], out[i]))
    res = sorted(res, key=lambda x: -x[1])
    return res

def array_to_image(arr):
    arr = arr.transpose(2,0,1)
    c = arr.shape[0]
    h = arr.shape[1]
    w = arr.shape[2]
    arr = (arr/255.0).flatten()
    data = c_array(c_float, arr)
    im = IMAGE(w,h,c,data)
    return im

def preprocess_cv_img(cvimg):
    cvimg = array_to_image(cvimg)
    rgbgr_image(cvimg)
    return cvimg
  
def detect(net, meta, image, thresh=.5, hier_thresh=.5, nms=.45):
    kk = cv2.imread(image)
    print(type(kk))
    im = array_to_image(kk) 
    num = c_int(0)
    pnum = pointer(num)
    predict_image(net, im)
    dets = get_network_boxes(net, im.w, im.h, thresh, hier_thresh, None, 0, pnum)
    num = pnum[0]
    if (nms): do_nms_obj(dets, num, meta.classes, nms);

    res = []
    for j in range(num):
        for i in range(meta.classes):
            if dets[j].prob[i] > 0:
                b = dets[j].bbox
                res.append((meta.names[i], dets[j].prob[i], (b.x, b.y, b.w, b.h)))
    res = sorted(res, key=lambda x: -x[1])
    free_detections(dets, num)
    return res

In [0]:
net = load_net("cfg/yolov3-tiny.cfg", "yolov3-tiny.weights", 0)
meta = load_meta("cfg/coco.data")
r = detect(net, meta, "data/dog.jpg")
print r

<type 'numpy.ndarray'>
[('dog', 0.5667503476142883, (224.1243896484375, 388.5039978027344, 214.8597412109375, 295.7761535644531)), ('truck', 0.557548463344574, (577.6723022460938, 124.36788940429688, 90.74085235595703, 63.93983459472656)), ('car', 0.51468825340271, (577.6723022460938, 124.36788940429688, 90.74085235595703, 63.93983459472656))]


In [0]:
def convert_to_coordinates(result): #once per bbox
        coords = list(map(lambda v: int(v), list(result[2])))
        x,y,w,h = coords #x and y of centre / anchor point
        
        pt1 = x-(w/2),y+(h/2) #left top corner (xmin,ymax)
        pt2 = x+(w/2),y-(h/2) #right bottom cornet
        return pt1,pt2

def fusecoordinates(coordinates_tuple):
  pt1,pt2 = coordinates_tuple
  return list(pt1)+list(pt2)

In [32]:
 [ fusecoordinates(convert_to_coordinates(re)) for re in r  ]

[[117, 535, 331, 241], [532, 155, 622, 93], [532, 155, 622, 93]]

In [0]:
labels_detected = [ e[0] for e in r ]
class_labels = ['car','person']
labels_matched = list(set(labels_detected) & set(class_labels))

In [27]:
labels_detected, class_labels, labels_matched

(['dog', 'truck', 'car'], ['car', 'person'], ['car'])

In [46]:
from ctypes import *
import math
import os,sys
import cv2
import numpy as np
import random

def import_file_utils():
  try:
    sys.path.append('..')
    global exec_cmd
    from fileutils import exec_cmd
  except Exception as e:
    print('Unable to import fileutils')

import_file_utils() # comment this line while running on notebooks

class BOX(Structure):
    _fields_ = [("x", c_float),
                ("y", c_float),
                ("w", c_float),
                ("h", c_float)]


class DETECTION(Structure):
    _fields_ = [("bbox", BOX),
                ("classes", c_int),
                ("prob", POINTER(c_float)),
                ("mask", POINTER(c_float)),
                ("objectness", c_float),
                ("sort_class", c_int)]


class IMAGE(Structure):
    _fields_ = [("w", c_int),
                ("h", c_int),
                ("c", c_int),
                ("data", POINTER(c_float))]


class METADATA(Structure):
    _fields_ = [("classes", c_int),
                ("names", POINTER(c_char_p))]

class yolo:
    def prepare(self):
        print('Preparing the environment')
        exec_cmd('git clone https://github.com/manuhg/darknet '+self.env_dir+self.name)
        exec_cmd('cd '+self.env_dir+self.name+' && make -j8 && cp libdarknet* ../')
        if not os.path.isfile(self.model['weights']):
            exec_cmd('wget -nc https://pjreddie.com/media/files/'+self.model_name+'.weights -o '+self.pretrained_models_dir+self.model_name+'.weights')
    
    def load_shared_lib(self,path=None):
        path  = self.shared_lib_path if path is None else path
        if not os.path.isfile(path):
            print(path,' not found')
            return False
        try:
            self.lib = CDLL(self.env_dir+"libdarknet.so", RTLD_GLOBAL)
            print('Imported libdarknet',self.lib)
            return True
        except Exception as e:
            print('Error loading shared library ',e)
        return False
    
    def c_array(self,ctype, values):
        arr = (ctype*len(values))()
        arr[:] = values
        return arr

    def array_to_image(self,arr):
        arr = arr.transpose(2,0,1)
        c = arr.shape[0]
        h = arr.shape[1]
        w = arr.shape[2]
        arr = (arr/255.0).flatten()
        data = self.c_array(c_float, arr)
        im = IMAGE(w,h,c,data)
        return im
    
    def preprocess_cv_img(self,cvimg):
        cvimg = self.array_to_image(cvimg)
        #self.rgbgr_image(cvimg)
        return cvimg
    
    def convert_to_coordinates(self,result): #once per bbox
        coords = list(map(lambda v: int(v), list(result[2])))
        x,y,w,h = coords #x and y of centre / anchor point
        
        pt1 = x-(w/2),y+(h/2) #left top corner (xmin,ymax)
        pt2 = x+(w/2),y-(h/2) #right bottom cornet
        return pt1,pt2

    def box_and_label(self,result, img,colors): #once per bbox
        #coords = list(map(lambda v: int(v), list(result[2])))
        #x,y,w,h = coords #x and y of centre / anchor point
        
        #pt1 = x-(w/2),y+(h/2) #left top corner (xmin,ymax)
        #pt2 = x+(w/2),y-(h/2) #right bottom cornet
        pt1,pt2 = self.convert_to_coordinates(result)

        label = result[0] + ' - '+str(np.around(float(result[1]), decimals=2))
        color = random.choice(colors)
        cv2.rectangle(img, pt1, pt2, color, 1)
        t_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_PLAIN, 1, 1)[0]
        pt2 = pt1[0] + t_size[0] + 3, pt1[1] + t_size[1] + 4
        cv2.rectangle(img, pt1, pt2, color, -1)
        cv2.putText(img, label, (pt1[0], pt1[1] + t_size[1] + 4),cv2.FONT_HERSHEY_PLAIN, 1, [225, 255, 255], 1)
        return img
    
    def detect_(self,net, meta, image, output_file='predictions.jpg', thresh=.5, hier_thresh=.5, nms=.45,visualize=False):
        h = self.lib.network_height(net)
        w = self.lib.network_width(net)

        if type(image)==str:
            image=cv2.imread(image)
            #im = load_image(image, 0, 0)
        #else:
        im = self.preprocess_cv_img(image)
        #im_sized = letterbox_image(im, h, w)
        im_sized=im

        num = c_int(0)
        pnum = pointer(num)
        self.predict_image(net, im_sized)
        dets = self.get_network_boxes(net, im.w, im.h, thresh, hier_thresh, None, 0, pnum)
        num = pnum[0]

        if (nms):
            self.do_nms_obj(dets, num, meta.classes, nms)
        
        res = []
        for j in range(num):
            for i in range(meta.classes):
                if dets[j].prob[i] > 0:
                    b = dets[j].bbox
                    res.append((meta.names[i], dets[j].prob[i], (b.x, b.y, b.w, b.h)))
        res = sorted(res, key=lambda x: -x[1])
        if visualize:
            for r in res:
                image=self.box_and_label(r,image,self.colors)
        cv2.imwrite(output_file,image)
        self.free_image(im)
        self.free_detections(dets, num)
        return res
    
    def load(self):
        try:
            print('Loading net..',self.model['cfg'], self.model['weights'])
            self.net = self.load_net(self.model['cfg'], self.model['weights'], 0)
            print('Loading metadata')
            self.meta = self.load_meta(self.env_dir+"cfg/coco.data")
            return True
        except Exception as e:
            print('Error loading network',e)
        return False

    def fusecoordinates(self,coordinates_tuple):
        pt1,pt2 = coordinates_tuple
        return list(pt1)+list(pt2)

    def detect(self,image,opfile,class_labels_to_filter, visualize=False):
        result  = self.detect_(self.net, self.meta, image, opfile)
        labels_detected = [ r[0] for r in result ]
        labels_matched = list(set(labels_detected) & set(class_labels_to_filter))
        bbox_converted =  [ self.fusecoordinates(self.convert_to_coordinates(r)) for r in result  ]
        output_dict = {'bounding_boxes': bbox_converted,'labels_detected':labels_detected}
        return {'labels_matched': labels_matched, 'output': output_dict}

    def __init__(self,model_name='yolov2',prepared = False,env_parent='models/env/'):
        self.name='YOLO'
        self.env_dir = 'env_'+self.name+'/'
        self.pretrained_models_dir = 'pretrained/'

        if env_parent:
            self.env_dir = env_parent + self.env_dir
            self.pretrained_models_dir = env_parent + self.pretrained_models_dir

        self.shared_lib_path = self.env_dir+"libdarknet.so"
        self.lib = None
        self.prepared = prepared
        self.model_name = model_name


        self.weights_base_url = 'https://pjreddie.com/media/files/'
        self.cfg_base_url = 'https://raw.githubusercontent.com/pjreddie/darknet/master/'
        self.models_lst = ['yolov2', 'yolov2-tiny', 'yolov3', 'yolov3-tiny']
        self.models = {}
        for model_name_ in self.models_lst:
            self.models.update({model_name_: {'name': model_name_, 'cfg': self.env_dir+'cfg/' +model_name_+'.cfg', 'weights': self.env_dir+model_name_ + '.weights'}})
        self.model_name = model_name
        self.model = self.models[self.model_name]

        
        
        exec_cmd('mkdir -p '+self.env_dir)
        exec_cmd('mkdir -p '+self.pretrained_models_dir)
        
        self.colors = [(39, 129, 113), (164, 80, 133), (83, 122, 114), (99, 81, 172), (95, 56, 104), (37, 84, 86), (14, 89, 122),
        (80, 7, 65), (10, 102, 25),(90, 185, 109), (106, 110, 132), (169, 158, 85), (188, 185, 26), (103, 1, 17), (82, 144, 81), 
        (92, 7, 184), (49, 81, 155), (179, 177, 69), (93, 187, 158), (13, 39, 73), (12, 50, 60), (16, 179, 33), (112, 69, 165), 
        (15, 139, 63), (33, 191, 159), (182, 173, 32), (34, 113, 133), (90, 135, 34), (53, 34, 86), (141, 35, 190), (6, 171, 8), 
        (118, 76, 112), (89, 60, 55), (15, 54, 88), (112, 75, 181), (42, 147, 38), (138, 52, 63), (128, 65, 149), (106, 103, 24), 
        (168, 33, 45), (28, 136, 135), (86, 91, 108), (52, 11, 76), (142, 6, 189), (57, 81, 168), (55, 19, 148), (182, 101, 89), 
        (44, 65, 179), (1, 33, 26), (122, 164, 26), (70, 63, 134), (137, 106, 82), (120, 118, 52), (129, 74, 42), (182, 147, 112),
        (22, 157, 50), (56, 50, 20), (2, 22, 177), (156, 100, 106), (21, 35, 42), (13, 8, 121), (142, 92, 28), (45, 118, 33), 
        (105, 118, 30), (7, 185, 124), (46, 34, 146), (105, 184, 169), (22, 18, 5), (147, 71, 73), (181, 64, 91), (31, 39, 184),
        (164, 179, 33), (96, 50, 18), (95, 15, 106), (113, 68, 54), (136, 116, 112), (119, 139, 130), (31, 139, 34), (66, 6, 127),
        (62, 39, 2), (49, 99, 180), (49, 119, 155), (153, 50, 183), (125, 38, 3), (129, 87, 143), (49, 87, 40), (128, 62, 120),
        (73, 85, 148), (28, 144, 118), (29, 9, 24), (175, 45, 108), (81, 175, 64), (178, 19, 157), (74, 188, 190), (18, 114, 2),
        (62, 128, 96), (21, 3, 150), (0, 6, 95), (2, 20, 184), (122, 37, 185)]
        
        prepared = self.load_shared_lib()
        if not prepared:
            print('Need to prepare environment')
            self.prepare()
            prepared = self.load_shared_lib()
        
        lib = self.lib
        
        lib.network_width.argtypes = [c_void_p]
        lib.network_width.restype = c_int
        lib.network_height.argtypes = [c_void_p]
        lib.network_height.restype = c_int

        #load_alphabet,draw_detections,save_image,    letterbox_image
        load_alphabet = lib.load_alphabet
        load_alphabet.argtypes = []
        load_alphabet.restype = POINTER(POINTER(IMAGE))
        
        # self.draw_detections = lib.draw_detections
        # self.draw_detections.argtypes = [IMAGE, POINTER(DETECTION), c_int, c_float, POINTER(c_char_p), POINTER(POINTER(IMAGE)), c_int]
        # self.draw_detections.restype = IMAGE
        
        self.save_image = lib.save_image
        self.save_image.argtypes = [IMAGE, c_char_p]
        
        self.predict = lib.network_predict
        self.predict.argtypes = [c_void_p, POINTER(c_float)]
        self.predict.restype = POINTER(c_float)
        
        self.set_gpu = lib.cuda_set_device
        self.set_gpu.argtypes = [c_int]
        
        self.make_image = lib.make_image
        self.make_image.argtypes = [c_int, c_int, c_int]
        self.make_image.restype = IMAGE
        
        self.get_network_boxes = lib.get_network_boxes
        self.get_network_boxes.argtypes = [c_void_p, c_int, c_int,c_float, c_float, POINTER(c_int), c_int, POINTER(c_int)]
        self.get_network_boxes.restype = POINTER(DETECTION)

        self.make_network_boxes = lib.make_network_boxes
        self.make_network_boxes.argtypes = [c_void_p]
        self.make_network_boxes.restype = POINTER(DETECTION)

        self.free_detections = lib.free_detections
        self.free_detections.argtypes = [POINTER(DETECTION), c_int]

        self.free_ptrs = lib.free_ptrs
        self.free_ptrs.argtypes = [POINTER(c_void_p), c_int]

        self.network_predict = lib.network_predict
        self.network_predict.argtypes = [c_void_p, POINTER(c_float)]

        self.reset_rnn = lib.reset_rnn
        self.reset_rnn.argtypes = [c_void_p]

        self.load_net = lib.load_network
        self.load_net.argtypes = [c_char_p, c_char_p, c_int]
        self.load_net.restype = c_void_p

        self.do_nms_obj = lib.do_nms_obj
        self.do_nms_obj.argtypes = [POINTER(DETECTION), c_int, c_int, c_float]

        self.do_nms_sort = lib.do_nms_sort
        self.do_nms_sort.argtypes = [POINTER(DETECTION), c_int, c_int, c_float]

        self.free_image = lib.free_image
        self.free_image.argtypes = [IMAGE]

        # self.letterbox_image = lib.letterbox_image
        # self.letterbox_image.argtypes = [IMAGE, c_int, c_int]
        # self.letterbox_image.restype = IMAGE

        self.load_meta = lib.get_metadata
        self.lib.get_metadata.argtypes = [c_char_p]
        self.lib.get_metadata.restype = METADATA

        self.load_image = lib.load_image_color
        self.load_image.argtypes = [c_char_p, c_int, c_int]
        self.load_image.restype = IMAGE

        self.rgbgr_image = lib.rgbgr_image
        self.rgbgr_image.argtypes = [IMAGE]

        self.predict_image = lib.network_predict_image
        self.predict_image.argtypes = [c_void_p, IMAGE]
        self.predict_image.restype = POINTER(c_float)
    

Unable to import fileutils


In [47]:
y = yolo()



('Imported libdarknet', <CDLL 'models/env/env_YOLO/libdarknet.so', handle 560552c5f700 at 7f912acf0b50>)


In [0]:
y.load()